# User + Wallclock MoE with Decay Tuning

Takes the winning configuration from the 8-experiment comparison (User MoE + Wallclock MoE)
and tests the effect of different time-decay rates with a 120-day training lookback.

**Configuration:**
- Power users (top 1%) get own expert, further split by wallclock bin
- Non-power users share per-wallclock-bin models
- XGBoost Tuned (200 trees, depth 12)
- 120 windows × 6h, 120-day training lookback
- Test decay rates: 0 (flat), 0.01, 0.03, 0.05, 0.1

**Dataset:** NLR Kestrel, full benchmark window

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.experimental.xgboost_tuned_model import (
    ExperimentalXGBoostTunedConfig, ExperimentalXGBoostTunedModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load data with enough history for 120-day lookback
# Test coverage: 120 windows x 6h = 30 days ending at 2025-06-26
# First test split needs 120 days of training data before it
# So load from at least 2025-06-26 - 30d(test) - 120d(lookback) = ~2025-01-27
table = pq.read_table(DATA_PATH)
# Use a generous start to ensure all windows have full 120-day lookback
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)  # 177 days before window end
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows_all = df.to_dict('records')
print(f'Loaded: {len(rows_all):,} rows')
print(f'Date range: {df["submit_time"].min().date()} to {df["submit_time"].max().date()}')
span_days = (df['submit_time'].max() - df['submit_time'].min()).days
print(f'Span: {span_days} days (need >= 150 for 120-day lookback + 30-day test)')


## 2. Configuration

In [ ]:
N_WINDOWS = 120
TEST_WINDOW_HOURS = 6
TRAINING_LOOKBACK_DAYS = 120

# Decay rates to test
DECAY_RATES = [0.0, 0.01, 0.03, 0.05, 0.1]

# Power user: top 1%
POWER_USER_PERCENTILE = 0.99

# Wallclock bins
BIN_EDGES_H = [0, 2, 4, 24, 48, float('inf')]
BIN_LABELS = ['<=2h', '2-4h', '4-24h', '24-48h', '>48h']

print(f'Rolling eval: {N_WINDOWS} windows x {TEST_WINDOW_HOURS}h = {N_WINDOWS*TEST_WINDOW_HOURS/24:.0f} days test')
print(f'Training lookback: {TRAINING_LOOKBACK_DAYS} days')
print(f'Decay rates to test: {DECAY_RATES}')
print(f'Wallclock bins: {BIN_LABELS}')

## 3. Helpers

In [ ]:
def assign_wallclock_bin(row):
    wc_h = (row.get('requested_seconds') or 0) / 3600
    for i in range(len(BIN_EDGES_H) - 1):
        if wc_h <= BIN_EDGES_H[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]


def identify_power_users(rows):
    user_counts = Counter(row.get('user') for row in rows)
    if not user_counts:
        return set()
    threshold = np.percentile(list(user_counts.values()), POWER_USER_PERCENTILE * 100)
    return {user for user, count in user_counts.items() if count >= threshold}


def run_user_wc_moe(rows_all, power_users, decay_rate):
    """Run the full User MoE + Wallclock MoE with a given decay rate."""
    config_kwargs = dict(
        n_windows=N_WINDOWS,
        test_window_hours=TEST_WINDOW_HOURS,
        training_lookback_days=TRAINING_LOOKBACK_DAYS,
        max_svd_components=256,
        target_max_one_hot_width=2048,
        random_state=42,
        n_estimators=200,
        max_depth=12,
        learning_rate=0.03,
        min_child_weight=5,
        gamma=0.1,
        time_decay_rate=decay_rate,
    )
    
    all_payloads = []
    
    # Power users: per-user, per-wallclock-bin
    for user in sorted(power_users):
        user_rows = [r for r in rows_all if r.get('user') == user]
        user_bins = {}
        for r in user_rows:
            bl = assign_wallclock_bin(r)
            user_bins.setdefault(bl, []).append(r)
        
        for bl in BIN_LABELS:
            bin_rows = user_bins.get(bl, [])
            if len(bin_rows) >= 100:
                try:
                    model = ExperimentalXGBoostTunedModel(ExperimentalXGBoostTunedConfig(**config_kwargs))
                    payload = model.evaluate(bin_rows, capture_artifacts=True)
                    if payload['summary']['rows_scored'] > 0:
                        all_payloads.append(payload)
                        print(f'    power {user[:7]}/{bl}: scored={payload["summary"]["rows_scored"]:,}, MAE={payload["mae"]:,.0f}s')
                except Exception as e:
                    print(f'    power {user[:7]}/{bl}: FAILED — {e}')
    
    # Non-power users: per-wallclock-bin
    non_power_rows = [r for r in rows_all if r.get('user') not in power_users]
    np_bins = {}
    for r in non_power_rows:
        bl = assign_wallclock_bin(r)
        np_bins.setdefault(bl, []).append(r)
    
    for bl in BIN_LABELS:
        bin_rows = np_bins.get(bl, [])
        if len(bin_rows) >= 100:
            try:
                model = ExperimentalXGBoostTunedModel(ExperimentalXGBoostTunedConfig(**config_kwargs))
                payload = model.evaluate(bin_rows, capture_artifacts=True)
                if payload['summary']['rows_scored'] > 0:
                    all_payloads.append(payload)
                    print(f'    non-power/{bl}: scored={payload["summary"]["rows_scored"]:,}, MAE={payload["mae"]:,.0f}s')
            except Exception as e:
                print(f'    non-power/{bl}: FAILED — {e}')
    
    # Combine
    all_true = []
    all_pred = []
    for p in all_payloads:
        if '_y_true' in p and '_y_pred' in p:
            all_true.extend(p['_y_true'])
            all_pred.extend(p['_y_pred'])
    
    if not all_true:
        return None
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    mae = np.mean(np.abs(all_true - all_pred))
    rmse = np.sqrt(np.mean((all_true - all_pred)**2))
    return {'mae': mae, 'rmse': rmse, 'scored': len(all_true)}


# Identify power users
power_users = identify_power_users(rows_all)
power_jobs = sum(1 for r in rows_all if r.get('user') in power_users)
print(f'\nPower users (top 1%): {len(power_users)}')
print(f'Power user jobs: {power_jobs:,} ({power_jobs/len(rows_all)*100:.1f}%)')
print(f'Non-power user jobs: {len(rows_all)-power_jobs:,}')

## 4. Run User+WC MoE at each decay rate

In [ ]:
results = {}

for decay_rate in DECAY_RATES:
    print(f'\n{"="*70}')
    print(f'DECAY RATE = {decay_rate}')
    if decay_rate == 0:
        print('(flat weighting — all training jobs equal)')
    else:
        print(f'(day 0=1.0, day 30={np.exp(-decay_rate*30):.2f}, day 60={np.exp(-decay_rate*60):.2f}, day 120={np.exp(-decay_rate*120):.2f})')
    print(f'{"="*70}')
    
    result = run_user_wc_moe(rows_all, power_users, decay_rate)
    if result:
        results[decay_rate] = result
        print(f'\n  COMBINED: MAE={result["mae"]:,.0f}s, RMSE={result["rmse"]:,.0f}s, scored={result["scored"]:,}')
    else:
        print(f'\n  FAILED — no predictions produced')

## 5. Results

In [ ]:
print('=' * 70)
print('RESULTS — USER+WC MoE WITH DIFFERENT DECAY RATES')
print('=' * 70)

flat_mae = results.get(0.0, {}).get('mae', 0)

print(f'\n{"Decay rate":<12} {"MAE":>10} {"RMSE":>10} {"Scored":>10} {"vs flat":>10} {"Weight at day 120":>18}')
print('-' * 75)
for rate, r in sorted(results.items()):
    vs = f'{(r["mae"]-flat_mae)/flat_mae*100:+.1f}%' if rate > 0 else 'BASELINE'
    w120 = f'{np.exp(-rate*120):.3f}' if rate > 0 else '1.000'
    print(f'{rate:<12.2f} {r["mae"]:>10,.0f}s {r["rmse"]:>10,.0f}s {r["scored"]:>10,} {vs:>10} {w120:>18}')

best_rate = min(results.items(), key=lambda x: x[1]['mae'])
print(f'\nBest decay rate: {best_rate[0]} — MAE={best_rate[1]["mae"]:,.0f}s')

In [ ]:
# Plot
rates = sorted(results.keys())
maes = [results[r]['mae'] for r in rates]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rates, maes, 'o-', color='steelblue', markersize=10, linewidth=2)
ax.set_xlabel('Time decay rate')
ax.set_ylabel('MAE (seconds)')
ax.set_title('User+WC MoE: Effect of Time Decay Rate on MAE\n(120-day lookback, NLR Kestrel)')

for rate, mae in zip(rates, maes):
    ax.annotate(f'{mae:,.0f}s', (rate, mae), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=10)

ax.axhline(results[0.0]['mae'], color='red', linestyle='--', alpha=0.5,
           label=f'Flat weighting: {results[0.0]["mae"]:,.0f}s')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)
best_rate, best_r = min(results.items(), key=lambda x: x[1]['mae'])
print(f'\nBest decay rate: {best_rate}')
print(f'  MAE:  {best_r["mae"]:,.0f}s')
print(f'  RMSE: {best_r["rmse"]:,.0f}s')
if best_rate == 0:
    print('  Time-decay does not help — flat weighting is optimal.')
    print('  With a 120-day lookback, older jobs are still relevant.')
else:
    improvement = (best_r['mae'] - results[0.0]['mae']) / results[0.0]['mae'] * 100
    print(f'  vs flat: {improvement:+.1f}% MAE')
    print(f'  At this rate, a job from 120 days ago gets weight {np.exp(-best_rate*120):.3f}')
    print(f'  (vs 1.0 for a job from yesterday)')